In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

FYP_ROOT = Path("/path/to/BrainWear_Kareem/FYP")
if str(FYP_ROOT) not in sys.path:
    sys.path.insert(0, str(FYP_ROOT))

LEADERBOARD_PATH = FYP_ROOT / "aacbr" / "leaderboard.json"
print(f"Leaderboard: {LEADERBOARD_PATH}")
print(f"Exists: {LEADERBOARD_PATH.exists()}")

In [2]:
from aacbr.results_logger import load_leaderboard

df = load_leaderboard(LEADERBOARD_PATH)
print(f"{len(df)} runs in leaderboard")
df.columns.tolist()

In [3]:
# ── Summary table: one row per run, key metrics only ─────────────────────────
SUMMARY_COLS = [
    "run_id", "timestamp", "eval_script",
    "config.checkpoint",                          # present in trained-model runs
    "config.char_model", "config.n_bins", "config.agg_mode",
    "config.strategy", "config.strict",
    "config.resection", "config.score_name",
    "config.seed", "config.train_frac",
    "data_stats.n_train", "data_stats.n_test",
    "metrics.accuracy", "metrics.ordinal_mae",
    "metrics.per_class.macro avg.f1-score",
    "metrics.per_class.weighted avg.f1-score",
    "notes",
]

present = [c for c in SUMMARY_COLS if c in df.columns]
summary = df[present].copy()

# Round floats for readability
for col in ["metrics.accuracy", "metrics.ordinal_mae",
            "metrics.per_class.macro avg.f1-score",
            "metrics.per_class.weighted avg.f1-score"]:
    if col in summary.columns:
        summary[col] = summary[col].round(4)

# Deduplicate: keep the most recent entry for each unique config
DEDUP_COLS = [c for c in [
    "eval_script", "config.checkpoint",
    "config.char_model", "config.n_bins",
    "config.strategy", "config.strict", "config.seed",
    "config.agg_mode", "config.resection",
] if c in summary.columns]

summary = (
    summary
    .sort_values("timestamp", ascending=False)   # most recent first
    .drop_duplicates(subset=DEDUP_COLS, keep="first")
    .sort_values("metrics.accuracy", ascending=False)
    .reset_index(drop=True)
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
summary

In [4]:
# ── Detail view: pick a run_id to inspect its confusion matrix and per-class report ──
INSPECT_RUN_ID = summary.iloc[0]["run_id"]  # change to any run_id string

with open(LEADERBOARD_PATH) as f:
    records = json.load(f)

record = next((r for r in records if r["run_id"] == INSPECT_RUN_ID), None)
if record is None:
    print(f"run_id {INSPECT_RUN_ID!r} not found")
else:
    print(f"run_id : {record['run_id']}")
    print(f"time   : {record['timestamp']}")
    print(f"script : {record['eval_script']}")
    print(f"config : {record['config']}")
    print(f"data   : {record['data_stats']}")
    print(f"notes  : {record.get('notes', '')}")
    print()
    print(f"Accuracy  : {record['metrics']['accuracy']:.4f}")
    print(f"Ordinal MAE: {record['metrics']['ordinal_mae']:.4f}")
    print()
    print("Confusion matrix (rows=true, cols=pred):")
    print(np.array(record["metrics"]["confusion_matrix"]))
    print()
    print("Per-class metrics:")
    per = record["metrics"]["per_class"]
    print(pd.DataFrame(per).T.round(3))

In [5]:
# ── Bar chart: accuracy and MAE across runs ───────────────────────────────────
import matplotlib.pyplot as plt

if len(summary) == 0:
    print("No runs to plot.")
else:
    labels = [
        f"{row['eval_script'].replace('eval_', '')}\n"
        f"{row.get('config.char_model', '')} {row.get('config.n_bins', '')}bins"
        for _, row in summary.iterrows()
    ]

    fig, axes = plt.subplots(1, 2, figsize=(max(8, len(summary) * 1.2), 4))

    axes[0].bar(range(len(summary)), summary["metrics.accuracy"], color="steelblue")
    axes[0].set_xticks(range(len(summary)))
    axes[0].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    axes[0].set_ylabel("Accuracy")
    axes[0].set_title("Accuracy (↑)")

    axes[1].bar(range(len(summary)), summary["metrics.ordinal_mae"], color="coral")
    axes[1].set_xticks(range(len(summary)))
    axes[1].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    axes[1].set_ylabel("Ordinal MAE")
    axes[1].set_title("Ordinal MAE (↓)")

    plt.tight_layout()
    plt.show()

In [9]:
# ── Bar charts: best accuracy per strategy, one subplot per n_bins value ──────
import matplotlib.pyplot as plt
import numpy as np

summary["dim"] = summary["eval_script"].apply(lambda s: "3D" if "3d" in s.lower() else "2D")

n_bins_vals = sorted(summary["config.n_bins"].dropna().unique())
strategies  = sorted(summary["config.strategy"].dropna().unique())

COLOR_2D = "steelblue"
COLOR_3D = "coral"
WIDTH = 0.35

fig, axes = plt.subplots(len(n_bins_vals), 1,
                         figsize=(max(5, 4 * len(strategies)), 5 * len(n_bins_vals)),
                         squeeze=False)

for ax, n_bins in zip(axes[:, 0], n_bins_vals):
    sub = summary[summary["config.n_bins"] == n_bins]
    best = (
        sub.groupby(["config.strategy", "dim"])["metrics.per_class.macro avg.f1-score"]
        .max()
        .reset_index()
    )

    x = np.arange(len(strategies))
    for i, (dim, color) in enumerate([("2D", COLOR_2D), ("3D", COLOR_3D)]):
        dim_rows = best[best["dim"] == dim].set_index("config.strategy")
        heights = [
            dim_rows.loc[s, "metrics.per_class.macro avg.f1-score"] if s in dim_rows.index else float("nan")
            for s in strategies
        ]
        bars = ax.bar(x + (i - 0.5) * WIDTH, heights, WIDTH, label=dim, color=color)
        for bar, h in zip(bars, heights):
            if not np.isnan(h):
                ax.text(bar.get_x() + bar.get_width() / 2, h + 0.01,
                        f"{h:.3f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(strategies, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Best Macro F1")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"n_bins = {int(n_bins)}")
    ax.axhline(1 / n_bins, color="grey", linestyle=":", linewidth=1.2, label=f"chance (1/{int(n_bins)}"+")")
    ax.legend()

plt.suptitle("Best macro-averaged F1 per strategy, grouped by n_bins (2D vs 3D)", fontsize=13)
plt.tight_layout()
plt.show()
